# How small can you go?

Every aggregation trades **size** for **accuracy**, across two levers: the number of typical
**periods** and the number of **segments** per period. This page picks them — first by hand, then
by search.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import SegmentConfig
from tsam.tuning import find_optimal_combination, find_pareto_front

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## By hand: the diminishing-returns curve

Sweeping the number of periods gives the classic shape — error drops fast, then flattens. The
"knee" is where extra periods stop paying off.

In [ ]:
rows = []
for k in [2, 4, 6, 8, 12, 16, 24]:
    r = tsam.aggregate(data, n_clusters=k, period_duration="1D")
    rows.append(
        {
            "n_clusters": k,
            "timesteps": k * r.n_timesteps_per_period,
            "rmse": float(r.accuracy.rmse.mean()),
        }
    )
px.line(
    pd.DataFrame(rows),
    x="timesteps",
    y="rmse",
    markers=True,
    title="Accuracy vs. size — number of periods",
)

## Why two levers, not one

Fix the budget at **48 time steps** (`n_clusters × n_segments`). You could keep 2 days at full
24-hour resolution, or 12 days of 4 segments each. Same budget, very different accuracy:

In [ ]:
rows = []
for n_clusters, n_segments in [
    (2, 24),
    (4, 12),
    (6, 8),
    (8, 6),
    (12, 4),
    (16, 3),
    (24, 2),
]:
    r = tsam.aggregate(
        data,
        n_clusters=n_clusters,
        period_duration="1D",
        segments=SegmentConfig(n_segments=n_segments),
    )
    rows.append(
        {
            "split": f"{n_clusters}d x {n_segments}seg",
            "rmse": float(r.accuracy.rmse.mean()),
        }
    )
px.bar(
    pd.DataFrame(rows),
    x="split",
    y="rmse",
    title="Same 48-step budget, different day/segment splits",
)

Both extremes lose, and the sweet spot moves with the budget — which is the chore the search
below removes.

## Hit a target reduction

Give [`find_optimal_combination`](../reference/api/tuning.md) a target data reduction and it
returns the most accurate period/segment combination that fits:

In [ ]:
best = find_optimal_combination(
    data,
    data_reduction=0.1,  # keep ~10% of the time steps
    period_duration="1D",
    n_jobs=1,
    show_progress=False,
)
print(f"best within budget: {best.n_clusters} periods x {best.n_segments} segments")
print(f"  time steps: {best.n_clusters * best.n_segments}  |  RMSE: {best.rmse:.4f}")

## Map the whole frontier

To choose deliberately instead of against one fixed budget,
[`find_pareto_front`](../reference/api/tuning.md) finds the best aggregation at each size.

`timesteps` sets the budgets to search. Leave it out and tsam sweeps every budget it can form,
which on this data takes minutes; naming the budgets you care about is both faster and easier to
read. Computed once here — everything below reuses it.

In [ ]:
pareto = find_pareto_front(
    data,
    period_duration="1D",
    timesteps=range(24, 721, 24),
    n_jobs=2,
    numerical_tolerance=1e-8,
    show_progress=False,
)
pareto.plot()

## How the search spends the budget

Not just *how accurate* at each size, but *how* — it adds periods first, then reaches for an extra
segment as the budget grows:

In [ ]:
mix = pareto.summary.melt(
    id_vars="timesteps",
    value_vars=["n_clusters", "n_segments"],
    var_name="lever",
    value_name="count",
)
mix["lever"] = mix["lever"].map({"n_clusters": "periods", "n_segments": "segments"})
px.bar(
    mix,
    x="timesteps",
    y="count",
    color="lever",
    barmode="group",
    title="How the search splits each budget",
)

## Pull a result off the frontier

`find_by_timesteps` and `find_by_rmse` both return an ordinary `AggregationResult`, ready to hand
to a model:

In [ ]:
small = pareto.find_by_timesteps(36)
print(
    f"at 36 steps: {small.n_clusters} periods x {small.n_segments} segments, "
    f"RMSE {small.accuracy.rmse.mean():.4f}"
)

accurate = pareto.find_by_rmse(0.10)  # smallest aggregation under an RMSE target
print(
    f"under RMSE 0.10: {accurate.n_clusters} periods x {accurate.n_segments} segments, "
    f"{accurate.n_clusters * accurate.n_segments} steps"
)

## Watch the detail go

Every point on the frontier as one animation. Each frame stacks all four variables (normalised) as
a day-by-hour heatmap of the reconstruction, starting at the **finest** resolution. As the budget
shrinks, the daily and seasonal structure blurs away.

In [ ]:
period_duration = pareto.best_result.n_timesteps_per_period  # 24
n_days = len(data) // period_duration
n_vars = len(data.columns)
data_min, data_range = data.min(), data.max() - data.min()

frames, labels = [], []
for r in sorted(
    pareto.all_results, key=lambda r: r.n_clusters * r.n_segments, reverse=True
):
    reduction = 1 - (r.n_clusters * r.n_segments) / len(data)
    labels.append(f"{reduction:.0%} smaller ({r.n_clusters}d x {r.n_segments}seg)")
    norm = (r.reconstructed - data_min) / data_range
    arr = norm.values.reshape(n_days, period_duration, n_vars).transpose(2, 1, 0)
    frames.append(arr.reshape(-1, n_days))

fig = px.imshow(
    np.stack(frames),
    animation_frame=0,
    aspect="auto",
    color_continuous_scale="RdYlBu_r",
    labels={"x": "day", "y": "hour"},
    title="Time series aggregation",
)
for i, step in enumerate(fig.layout.sliders[0].steps):
    step["label"] = labels[i]
ticks = [period_duration * i + period_duration // 2 for i in range(n_vars)]
fig.update_yaxes(tickvals=ticks, ticktext=list(data.columns))
fig.update_layout(height=600, coloraxis_showscale=False)
fig

---

* [Segmentation](segmentation.ipynb) — the second lever on its own.
* [How long will this take?](runtime.ipynb) — a search runs many aggregations; budget accordingly.
* [Optimization workflow](optimization_workflow.ipynb) — handing a chosen point to a model.